In [5]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
import torch
from PIL import Image
from diffsynth import save_video, VideoData, load_state_dict
from diffsynth.pipelines.wan_video_new import WanVideoPipeline, ModelConfig
from modelscope import dataset_snapshot_download
import numpy as np
from utils import process_bracketed_video
from utils import average_frame_psnr



pipe = WanVideoPipeline.from_pretrained(
    torch_dtype=torch.bfloat16,
    device="cuda:7",
    model_configs=[
        ModelConfig(model_id="Wan-AI/Wan2.2-TI2V-5B", origin_file_pattern="models_t5_umt5-xxl-enc-bf16.pth", offload_device="cpu", skip_download=True),
        ModelConfig(model_id="Wan-AI/Wan2.2-TI2V-5B", origin_file_pattern="diffusion_pytorch_model*.safetensors", offload_device="cpu", skip_download=True),
        ModelConfig(model_id="Wan-AI/Wan2.2-TI2V-5B", origin_file_pattern="Wan2.2_VAE.pth", offload_device="cpu", skip_download=True),
    ],
)

pipe.load_models_to_device(["vae"])

# ensure the VAE is actually on the GPU in the right dtype
pipe.vae = pipe.vae.to(device=pipe.device)
pipe.vae.eval()  # optional but good practice

Loading models from: ./models/Wan-AI/Wan2.2-TI2V-5B/models_t5_umt5-xxl-enc-bf16.pth
    model_name: wan_video_text_encoder model_class: WanTextEncoder
    The following models are loaded: ['wan_video_text_encoder'].
Loading models from: ['./models/Wan-AI/Wan2.2-TI2V-5B/diffusion_pytorch_model-00001-of-00003-bf16.safetensors', './models/Wan-AI/Wan2.2-TI2V-5B/diffusion_pytorch_model-00002-of-00003-bf16.safetensors', './models/Wan-AI/Wan2.2-TI2V-5B/diffusion_pytorch_model-00003-of-00003-bf16.safetensors']
    model_name: wan_video_dit model_class: WanModel
        This model is initialized with extra kwargs: {'has_image_input': False, 'patch_size': [1, 2, 2], 'in_dim': 48, 'dim': 3072, 'ffn_dim': 14336, 'freq_dim': 256, 'text_dim': 4096, 'out_dim': 48, 'num_heads': 24, 'num_layers': 30, 'eps': 1e-06, 'seperated_timestep': True, 'require_clip_embedding': False, 'require_vae_embedding': False, 'fuse_vae_embedding_in_latents': True}
Initializing missing parameter: blocks.0.self_attn.all_info

WanVideoVAE38(
  (model): VideoVAE38_(
    (encoder): Encoder3d_38(
      (conv1): CausalConv3d(12, 160, kernel_size=(3, 3, 3), stride=(1, 1, 1))
      (downsamples): Sequential(
        (0): Down_ResidualBlock(
          (avg_shortcut): AvgDown3D()
          (downsamples): Sequential(
            (0): ResidualBlock(
              (residual): Sequential(
                (0): RMS_norm()
                (1): SiLU()
                (2): CausalConv3d(160, 160, kernel_size=(3, 3, 3), stride=(1, 1, 1))
                (3): RMS_norm()
                (4): SiLU()
                (5): Dropout(p=0.0, inplace=False)
                (6): CausalConv3d(160, 160, kernel_size=(3, 3, 3), stride=(1, 1, 1))
              )
              (shortcut): Identity()
            )
            (1): ResidualBlock(
              (residual): Sequential(
                (0): RMS_norm()
                (1): SiLU()
                (2): CausalConv3d(160, 160, kernel_size=(3, 3, 3), stride=(1, 1, 1))
                (3):

In [7]:
from diffsynth.trainers.stuttgart_dataset import StuttgartDataset
import numpy as np


dataset = StuttgartDataset(
    base_path="/data2/saikiran.tedla/hdrvideo/diff/data/stuttgart/poker_fullshot",
    repeat=1,
    main_data_operator=StuttgartDataset.default_video_operator(
        base_path="/data2/saikiran.tedla/hdrvideo/diff/data/stuttgart/poker_fullshot",
        max_pixels=1280*720,
        height=480,
        width=832,
        height_division_factor=16,
        width_division_factor=16,
        num_frames=15,
        time_division_factor=4,
        time_division_remainder=1,

    ),
    split='train',
    mode = "hdr_and_brackets"
)

data = dataset[0]
bracket_video = data["bracket_video"]
#convert bracket video (list of PIL images) to tensor
bracket_video = torch.stack([torch.from_numpy(np.array(img)).permute(2,0,1) for img in bracket_video], dim=0).float() / 255.0  # shape (T, C, H, W)
hdr_video = torch.from_numpy(data["hdr_video"])
hdr_video = hdr_video.permute(0,3,1,2)  # shape (C, T, H, W)
#reshape hdr_video to same H,W as bracket_video
hdr_video = torch.nn.functional.interpolate(hdr_video, size=bracket_video.shape[2:], mode="bilinear", align_corners=False).squeeze(0)
bracket_video = bracket_video.unsqueeze(0).to(pipe.device).to(torch.bfloat16) 
hdr_video = hdr_video.unsqueeze(0).to(pipe.device).to(torch.bfloat16)


normal_exposure = bracket_video[:, 17:34] #EV 0
low_exposure = bracket_video[:, 34:51] #EV -4
high_exposure = bracket_video[:, 51:68] #EV +4


print("bracket_video shape:", bracket_video.shape)


bracket_video shape: torch.Size([1, 68, 3, 480, 832])


In [8]:
from utils import output_frames, merge_hdr, pu21_encode_linear_video, pu21_decode_pu_video, PU21_IN_MAX_VALUE

def process_bracketed_video(video):
    #assert that video has 12 frames
    assert video.shape[0] == 51, "Video must have 15 frames for bracketed HDR processing"
    normal_exposure = video[0:17] #EV 0
    low_exposure = video[17:34] #EV -4
    high_exposure = video[34:51] #EV +4

    normal_radiance = normal_exposure
    low_radiance = low_exposure * (2 ** 4)  # EV -4
    high_radiance = high_exposure * (2 ** -4)  # EV +4

    hdr_video = merge_hdr(normal_exposure,low_exposure,high_exposure, normal_radiance, low_radiance,high_radiance)

    return hdr_video


def encode_and_decode(video):
    with torch.no_grad():
        video = video * 2.0 - 1.0  # scale to [-1, 1]
        video = video.permute(0,2,1,3,4)  # (B, T, C, H, W) to (B, C, T, H, W)
        input_latents = pipe.vae.encode(video, device=pipe.device, tiled=False).to(dtype=pipe.torch_dtype, device=pipe.device)
        output_video = pipe.vae.decode(input_latents, device=pipe.device, tiled=False).to(dtype=torch.float32, device="cpu")
        output_video = (output_video + 1.0) / 2.0  # scale to [0, 1]
        output_video = torch.clamp(output_video, 0.0, 1.0) #clip output video to [0,1]
        output_video = output_video.permute(0,2,1,3,4)  # (B, C, T, H, W) to (B, T, C, H, W)
    return output_video

decoded_normal_exposure = encode_and_decode(normal_exposure)
decoded_low_exposure = encode_and_decode(low_exposure)
decoded_high_exposure = encode_and_decode(high_exposure)
decoded_combined = torch.cat([decoded_normal_exposure, decoded_low_exposure, decoded_high_exposure], dim=1)
seperate_bracket_to_hdr_video = process_bracketed_video(decoded_combined[0].to(dtype=torch.float32, device="cpu")).unsqueeze(0) # with original bracket video


# encode and decode the hdr video (linear 01 -> VAE -> linear)
GAMMA = 2.2  # power-law compression exponent (gamma encoding before VAE)
max_hdr_value = torch.max(hdr_video)
decoded_hdr_video = encode_and_decode(hdr_video / max_hdr_value) * max_hdr_value.cpu()

# linear HDR -> gamma encode -> VAE encode/decode -> gamma decode -> linear HDR
linear_hdr_01 = (hdr_video / max_hdr_value).clamp(min=1e-10)
log_gamma_encoded = torch.pow(linear_hdr_01, 1.0 / GAMMA)
decoded_after_vae = encode_and_decode(log_gamma_encoded)
decoded_hdr_log_gamma = (
    torch.pow(decoded_after_vae.clamp(min=0.0), GAMMA) * max_hdr_value.cpu()
)

# linear HDR -> PU21 encode -> VAE -> PU21 decode -> linear HDR (same pattern as test_pu_encoder.ipynb)
scale_for_pu = 1000 / hdr_video.max().item()
pu_hdr = pu21_encode_linear_video(hdr_video * scale_for_pu)
pu_for_vae = (pu_hdr / PU21_IN_MAX_VALUE).clamp(0.0, 1.0)
decoded_pu_01 = encode_and_decode(pu_for_vae)
decoded_pu_native = decoded_pu_01 * PU21_IN_MAX_VALUE * (1/scale_for_pu)
decoded_hdr_pu = pu21_decode_pu_video(decoded_pu_native)

print(seperate_bracket_to_hdr_video.shape)
output_frames(seperate_bracket_to_hdr_video[0][:,:,10:-10, 10:-10], "/data2/saikiran.tedla/hdrvideo/diff/encoder_test/seperate_bracket_to_hdr_video", "hdr", channel_order="NCHW")
output_frames(decoded_hdr_video[0][:,:,10:-10,10:-10], "/data2/saikiran.tedla/hdrvideo/diff/encoder_test/decoded_hdr_video", "hdr", channel_order="NCHW")
output_frames(decoded_hdr_log_gamma[0][:,:,10:-10,10:-10], "/data2/saikiran.tedla/hdrvideo/diff/encoder_test/decoded_hdr_log_gamma_video", "hdr", channel_order="NCHW")
output_frames(decoded_hdr_pu[0][:,:,10:-10,10:-10], "/data2/saikiran.tedla/hdrvideo/diff/encoder_test/decoded_hdr_pu21_video", "hdr", channel_order="NCHW")
output_frames(hdr_video[0][:,:,10:-10,10:-10].to(dtype=torch.float32, device="cpu"), "/data2/saikiran.tedla/hdrvideo/diff/encoder_test/original_hdr_video", "hdr", channel_order="NCHW")

torch.Size([1, 17, 3, 480, 832])


In [9]:
from utils import write_exposure_bracket
evs = [-8, -6, -4, -2, 0, 2, 4, 6, 8]
write_exposure_bracket("/data2/saikiran.tedla/hdrvideo/diff/encoder_test/seperate_bracket_to_hdr_video/frame_0000.hdr", evs)
write_exposure_bracket("/data2/saikiran.tedla/hdrvideo/diff/encoder_test/decoded_hdr_video/frame_0000.hdr", evs)
write_exposure_bracket("/data2/saikiran.tedla/hdrvideo/diff/encoder_test/decoded_hdr_log_gamma_video/frame_0000.hdr", evs)
write_exposure_bracket("/data2/saikiran.tedla/hdrvideo/diff/encoder_test/decoded_hdr_pu21_video/frame_0000.hdr", evs)
write_exposure_bracket("/data2/saikiran.tedla/hdrvideo/diff/encoder_test/original_hdr_video/frame_0000.hdr", evs)

PosixPath('/data2/saikiran.tedla/hdrvideo/diff/encoder_test/original_hdr_video/bracket')